# 02 - Preprocessing & Augmentation Pipeline Demo

Demonstrates the training and validation augmentation pipelines
defined in `src/transforms.py` using Albumentations.

In [ ]:
import sys
sys.path.append("..")

import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.transforms import get_train_transforms, get_val_transforms, IMAGENET_MEAN, IMAGENET_STD

## 1. Load a Sample Image

In [ ]:
DATA_DIR = "../data/raw"
IMAGE_SIZE = 224

# Load a sample image (update the filename to an actual image in your dataset)
sample_files = [f for f in os.listdir(DATA_DIR) if f.endswith(".jpg")]
if sample_files:
    sample_path = os.path.join(DATA_DIR, sample_files[0])
    image = np.array(Image.open(sample_path).convert("RGB"))
    print(f"Loaded: {sample_files[0]}, shape: {image.shape}")
    plt.imshow(image)
    plt.title("Original Image")
    plt.axis("off")
    plt.show()
else:
    print("No images found in data/raw/. Download the ISIC 2019 dataset first.")
    image = np.random.randint(0, 255, (450, 600, 3), dtype=np.uint8)
    print("Using a random placeholder image for demo purposes.")

## 2. Training Augmentations

The training pipeline includes: resize, flips, rotation, shift/scale/rotate,
color jitter, coarse dropout, and ImageNet normalization.

In [ ]:
train_transform = get_train_transforms(IMAGE_SIZE)

def denormalize(tensor):
    """Convert normalized tensor back to displayable image."""
    img = tensor.numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

# Show 8 augmented versions of the same image
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax in axes.flat:
    augmented = train_transform(image=image)
    display_img = denormalize(augmented["image"])
    ax.imshow(display_img)
    ax.axis("off")

plt.suptitle("Training Augmentations (8 random samples from same image)", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Validation Transform

The validation pipeline only applies resize + normalization (no augmentation).

In [ ]:
val_transform = get_val_transforms(IMAGE_SIZE)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Original (resized)
resized = Image.fromarray(image).resize((IMAGE_SIZE, IMAGE_SIZE))
axes[0].imshow(resized)
axes[0].set_title("Original (resized)")
axes[0].axis("off")

# After val transform
val_result = val_transform(image=image)
display_img = denormalize(val_result["image"])
axes[1].imshow(display_img)
axes[1].set_title("After Validation Transform")
axes[1].axis("off")

plt.suptitle("Validation Pipeline (deterministic)", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Augmentation Comparison

Side-by-side view of individual augmentation effects.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

augmentations = {
    "HorizontalFlip": A.HorizontalFlip(p=1.0),
    "VerticalFlip": A.VerticalFlip(p=1.0),
    "RandomRotate90": A.RandomRotate90(p=1.0),
    "ColorJitter": A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=1.0),
    "ShiftScaleRotate": A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=1.0),
    "CoarseDropout": A.CoarseDropout(max_holes=8, max_height=24, max_width=24, p=1.0),
}

fig, axes = plt.subplots(1, len(augmentations) + 1, figsize=(20, 4))

# Original
resized_arr = np.array(Image.fromarray(image).resize((IMAGE_SIZE, IMAGE_SIZE)))
axes[0].imshow(resized_arr)
axes[0].set_title("Original")
axes[0].axis("off")

for idx, (name, aug) in enumerate(augmentations.items(), 1):
    pipeline = A.Compose([A.Resize(IMAGE_SIZE, IMAGE_SIZE), aug])
    result = pipeline(image=image)
    axes[idx].imshow(result["image"])
    axes[idx].set_title(name, fontsize=9)
    axes[idx].axis("off")

plt.suptitle("Individual Augmentation Effects", fontsize=14)
plt.tight_layout()
plt.show()